# 01 — PyTorch baselines (FP32)

Runs:
- FP32 (CUDA)
- FP16 (CUDA)

Optionally without reduced bit depth.

In [1]:
from pathlib import Path
import sys, os

# ---- Path setup (adjust if your repo layout differs) ----
PROJECT_ROOT = Path("..").resolve()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

In [2]:
# Purpose: Establish PyTorch FP32 baseline (and optional FP16 baseline) on ImageNet val.
import os
import json
from pathlib import Path
import pandas as pd

# project imports
from config import ExperimentConfig, with_overrides
from runner import run_experiment
from utils import load_runs, flatten_runs

pd.set_option("display.max_columns", 200)

In [3]:
# Baselines (PyTorch backend only)
base = ExperimentConfig(
    backend="pytorch",
    device="cuda",             
    batch_size=1,
    input_quant_bits=8,        
    seed=42,
    num_eval_batches=500
)


In [4]:
cfg32 = with_overrides(base, model_precision="fp32")

payload, tracker = run_experiment(cfg32, save_results_flag=True)

# quick peek
print(payload["run_id"], payload["status"], payload["results"].get("top1_acc"))

[data] Train: 124,395  Holdout-Val: 5,000  (num_classes=100, val_per_class=50, seed=42)
Evaluating on 500 batches (first 30 are warmup)...
  --- Warmup complete (30 batches) — starting metric collection ---
  Batch [40/500] Top-1: 90.00% | Top-5: 90.00% | Infer: 3.23 ms/batch
  Batch [50/500] Top-1: 90.00% | Top-5: 90.00% | Infer: 3.12 ms/batch
  Batch [60/500] Top-1: 93.33% | Top-5: 93.33% | Infer: 3.02 ms/batch
  Batch [70/500] Top-1: 95.00% | Top-5: 95.00% | Infer: 3.02 ms/batch
  Batch [80/500] Top-1: 96.00% | Top-5: 96.00% | Infer: 3.15 ms/batch
  Batch [90/500] Top-1: 96.67% | Top-5: 96.67% | Infer: 3.14 ms/batch
  Batch [100/500] Top-1: 95.71% | Top-5: 97.14% | Infer: 3.17 ms/batch
  Batch [110/500] Top-1: 93.75% | Top-5: 97.50% | Infer: 3.18 ms/batch
  Batch [120/500] Top-1: 92.22% | Top-5: 96.67% | Infer: 3.18 ms/batch
  Batch [130/500] Top-1: 93.00% | Top-5: 97.00% | Infer: 3.16 ms/batch
  Batch [140/500] Top-1: 92.73% | Top-5: 96.36% | Infer: 3.16 ms/batch
  Batch [150/500] 

In [5]:
# Always load from disk (the source of truth)
runs = load_runs("../runs", status="ok")
rows = flatten_runs(runs)
df = pd.DataFrame(rows)

# Filter to just this notebook’s scope (pytorch + no input quant)
df_baselines = df[
    (df["cfg.backend"] == "pytorch")
    & (df["cfg.input_quant_bits"] == 8)
    & (df["cfg.model_precision"].isin(["fp32"]))
].copy()

df_baselines[[
    "run_id",
    "cfg.backend",
    "cfg.device",
    "cfg.batch_size",
    "cfg.model_precision",
    "cfg.input_quant_bits",
    "res.top1_acc",
    "res.top5_acc",
    "res.infer_ms_avg",
    "res.throughput_infer_sps",
    "res.total_samples",
]].sort_values(["cfg.model_precision", "cfg.device"])

,run_id,cfg.backend,cfg.device,cfg.batch_size,cfg.model_precision,cfg.input_quant_bits,res.top1_acc,res.top5_acc,res.infer_ms_avg,res.throughput_infer_sps,res.total_samples
0,resnet18_pytorch_fp32_in8b_cuda_bs1,pytorch,cuda,1,fp32,8,87.021277,97.87234,3.199306,312.56778,470


In [6]:
cfg16 = with_overrides(base, model_precision="fp16")

payload, tracker = run_experiment(cfg16, save_results_flag=True)

# quick peek
print(payload["run_id"], payload["status"], payload["results"].get("top1_acc"))

[data] Train: 124,395  Holdout-Val: 5,000  (num_classes=100, val_per_class=50, seed=42)
Evaluating on 500 batches (first 30 are warmup)...
  --- Warmup complete (30 batches) — starting metric collection ---
  Batch [40/500] Top-1: 90.00% | Top-5: 90.00% | Infer: 3.72 ms/batch
  Batch [50/500] Top-1: 90.00% | Top-5: 90.00% | Infer: 3.24 ms/batch
  Batch [60/500] Top-1: 93.33% | Top-5: 93.33% | Infer: 3.03 ms/batch
  Batch [70/500] Top-1: 95.00% | Top-5: 95.00% | Infer: 3.06 ms/batch
  Batch [80/500] Top-1: 96.00% | Top-5: 96.00% | Infer: 3.02 ms/batch
  Batch [90/500] Top-1: 96.67% | Top-5: 96.67% | Infer: 2.93 ms/batch
  Batch [100/500] Top-1: 95.71% | Top-5: 97.14% | Infer: 2.96 ms/batch
  Batch [110/500] Top-1: 93.75% | Top-5: 97.50% | Infer: 2.98 ms/batch
  Batch [120/500] Top-1: 92.22% | Top-5: 96.67% | Infer: 2.97 ms/batch
  Batch [130/500] Top-1: 93.00% | Top-5: 97.00% | Infer: 3.03 ms/batch
  Batch [140/500] Top-1: 92.73% | Top-5: 96.36% | Infer: 3.02 ms/batch
  Batch [150/500] 

In [7]:
# Always load from disk (the source of truth)
runs = load_runs("../runs", status="ok")
rows = flatten_runs(runs)
df = pd.DataFrame(rows)

# Filter to just this notebook’s scope (pytorch + no input quant)
df_baselines = df[
    (df["cfg.backend"] == "pytorch")
    & (df["cfg.input_quant_bits"] == 8)
    & (df["cfg.model_precision"].isin(["fp16"]))
].copy()

df_baselines[[
    "run_id",
    "cfg.backend",
    "cfg.device",
    "cfg.batch_size",
    "cfg.model_precision",
    "cfg.input_quant_bits",
    "res.top1_acc",
    "res.top5_acc",
    "res.infer_ms_avg",
    "res.throughput_infer_sps",
    "res.total_samples",
]].sort_values(["cfg.model_precision", "cfg.device"])

,run_id,cfg.backend,cfg.device,cfg.batch_size,cfg.model_precision,cfg.input_quant_bits,res.top1_acc,res.top5_acc,res.infer_ms_avg,res.throughput_infer_sps,res.total_samples
0,resnet18_pytorch_fp16_in8b_cuda_bs1,pytorch,cuda,1,fp16,8,87.021277,97.87234,2.997852,333.572219,470


In [8]:
runs = load_runs("../runs", status="ok")
rows = flatten_runs(runs)
df = pd.DataFrame(rows)

df_baselines = df[
    (df["cfg.backend"] == "pytorch")
    & (df["cfg.input_quant_bits"] == 8)
    & (df["cfg.model_precision"].isin(["fp32", "fp16"]))
].copy()

df_baselines[[
    "run_id",
    "cfg.model_precision",
    "res.infer_ms_avg",
    "res.infer_ms_min",
    "res.infer_ms_max",
]].sort_values("cfg.model_precision")

,run_id,cfg.model_precision,res.infer_ms_avg,res.infer_ms_min,res.infer_ms_max
0,resnet18_pytorch_fp16_in8b_cuda_bs1,fp16,2.997852,1.108570,6.738398
1,resnet18_pytorch_fp32_in8b_cuda_bs1,fp32,3.199306,1.292127,7.958971
